<a href="https://colab.research.google.com/github/Argus-a/proj1-1/blob/data-for-the-proj/Copy_of_RObert_%D8%B9%D8%B1%D8%A8_%D8%A8%D8%A7%D8%B1%D8%AA_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from transformers import XLMRobertaForSequenceClassification, AutoConfig

# Specify the model name from Hugging Face
# Replace with your specific suicide detection model repository if it's hosted on Hugging Face
model_name = 'xlm-roberta-large'

# Load the configuration of the model
config = AutoConfig.from_pretrained(model_name)

# Print the label mappings
print(f"Model: {model_name}")
print(f"ID to Label mapping: {config.id2label}")
print(f"Label to ID mapping: {config.label2id}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Model: xlm-roberta-large
ID to Label mapping: {0: 'LABEL_0', 1: 'LABEL_1'}
Label to ID mapping: {'LABEL_0': 0, 'LABEL_1': 1}


In [4]:
# تنزيل البيانات من Kaggle وفك ضغطها في مجلد twitter_emotion_data
!kaggle datasets download -d aadyasingh55/twitter-emotion-classification-dataset --unzip -p twitter_emotion_data

print("تم تحميل البيانات وفك ضغطها بنجاح!")

Dataset URL: https://www.kaggle.com/datasets/aadyasingh55/twitter-emotion-classification-dataset
License(s): other
100% 22.3M/22.3M [00:00<00:00, 178MB/s]

تم تحميل البيانات وفك ضغطها بنجاح!


In [5]:
import pandas as pd

file_path = 'twitter_emotion_data/train-00000-of-00001.parquet'

print(f"Loading {file_path}...\n")
df_twitter_emotion = pd.read_parquet(file_path)

print(f"عدد الصفوف: {df_twitter_emotion.shape[0]}")
print(f"عدد الأعمدة: {df_twitter_emotion.shape[1]}")
print("\nالأعمدة المتوفرة:", df_twitter_emotion.columns.tolist())

# محاولة التعرف على عمود المشاعر
emotion_col = None
for col in ['emotion', 'sentiment', 'label', 'class', 'target']:
    if col in [c.lower() for c in df_twitter_emotion.columns]:
        emotion_col = [c for c in df_twitter_emotion.columns if c.lower() == col][0]
        break

if emotion_col:
    print(f"\nعدد الحالات لكل شعور (في عمود '{emotion_col}'):")
    display(df_twitter_emotion[emotion_col].value_counts())
else:
    print("\nلم أتمكن من تحديد عمود المشاعر تلقائياً. يرجى مراجعة البيانات أدناه:")
    display(df_twitter_emotion.head())


Loading twitter_emotion_data/train-00000-of-00001.parquet...

عدد الصفوف: 416809
عدد الأعمدة: 2

الأعمدة المتوفرة: ['text', 'label']

عدد الحالات لكل شعور (في عمود 'label'):


,count
label,
1,141067
0,121187
3,57317
4,47712
2,34554
5,14972


In [6]:
# قاموس لتسمية الفئات الرقمية بأسمائها الفعلية
label_mapping = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

# إنشاء عمود جديد يحتوي على أسماء المشاعر
df_twitter_emotion['emotion_name'] = df_twitter_emotion['label'].map(label_mapping)

print("توزيع المشاعر بالأسماء الجديدة:")
display(df_twitter_emotion['emotion_name'].value_counts())

print("\nأول 5 صفوف بعد إضافة الأسماء:")
display(df_twitter_emotion.head())

توزيع المشاعر بالأسماء الجديدة:


,count
emotion_name,
joy,141067
sadness,121187
anger,57317
fear,47712
love,34554
surprise,14972



أول 5 صفوف بعد إضافة الأسماء:


,text,label,emotion_name
0,i feel awful about it too because it s my job ...,0,sadness
1,im alone i feel awful,0,sadness
2,ive probably mentioned this before but i reall...,1,joy
3,i was feeling a little low few days back,0,sadness
4,i beleive that i am much more sensitive to oth...,2,love


In [7]:
import re

print(f"أبعاد البيانات قبل التنظيف: {df_twitter_emotion.shape}")

# 1. إزالة القيم المفقودة
df_twitter_emotion = df_twitter_emotion.dropna(subset=['text', 'label'])

# 2. إزالة الصفوف المكررة
df_twitter_emotion = df_twitter_emotion.drop_duplicates(subset=['text'])

# 3. دالة لتنظيف النصوص
def clean_tweet(text):
    if not isinstance(text, str):
        return str(text)
    # إزالة الروابط
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # إزالة الإشارات للمستخدمين
    text = re.sub(r'\@\w+', '', text)
    # إزالة الهاشتاجات (الرمز فقط)
    text = re.sub(r'#', '', text)
    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()
    # تحويل النص إلى أحرف صغيرة
    return text.lower()

# تطبيق دالة التنظيف
df_twitter_emotion['clean_text'] = df_twitter_emotion['text'].apply(clean_tweet)

# إزالة الصفوف التي أصبح نصها فارغاً بعد التنظيف
df_twitter_emotion = df_twitter_emotion[df_twitter_emotion['clean_text'] != '']

print(f"أبعاد البيانات بعد التنظيف: {df_twitter_emotion.shape}")
display(df_twitter_emotion[['text', 'clean_text', 'emotion_name']].head(10))

أبعاد البيانات قبل التنظيف: (416809, 3)
أبعاد البيانات بعد التنظيف: (393822, 4)


,text,clean_text,emotion_name
0,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,sadness
1,im alone i feel awful,im alone i feel awful,sadness
2,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,joy
3,i was feeling a little low few days back,i was feeling a little low few days back,sadness
4,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,love
5,i find myself frustrated with christians becau...,i find myself frustrated with christians becau...,love
6,i am one of those people who feels like going ...,i am one of those people who feels like going ...,joy
7,i feel especially pleased about this as this h...,i feel especially pleased about this as this h...,joy
8,i was struggling with these awful feelings and...,i was struggling with these awful feelings and...,joy
9,i feel so enraged but helpless at the same time,i feel so enraged but helpless at the same time,anger


### الخطوة 1: تقسيم البيانات إلى تدريب واختبار وتحويلها إلى Hugging Face Dataset

In [8]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# تقسيم البيانات (80% تدريب، 20% تقييم)
# نستخدم التقسيم الطبقي (stratify) للحفاظ على نسب الفئات
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_twitter_emotion['clean_text'].tolist(),
    df_twitter_emotion['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_twitter_emotion['label'].tolist()
)

# تحويلها إلى صيغة Dataset الخاصة بـ Hugging Face
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

print(f"عدد عينات التدريب: {len(train_dataset)}")
print(f"عدد عينات التقييم: {len(val_dataset)}")

عدد عينات التدريب: 315057
عدد عينات التقييم: 78765


### الخطوة 2: ترميز البيانات (Tokenization) باستخدام XLM-RoBERTa

In [9]:
from transformers import XLMRobertaTokenizer

# تحميل محلل النصوص (Tokenizer)
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-large')

def tokenize_function(examples):
    # تقطيع النصوص بحد أقصى 128 لتسريع العملية
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

# تطبيق التقطيع على البيانات
print("جاري تقطيع نصوص التدريب...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
print("جاري تقطيع نصوص التقييم...")
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# تحويل الأعمدة المهمة إلى صيغة PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

جاري تقطيع نصوص التدريب...


Map:   0%|          | 0/315057 [00:00<?, ? examples/s]

جاري تقطيع نصوص التقييم...


Map:   0%|          | 0/78765 [00:00<?, ? examples/s]

### الخطوة 3: حساب أوزان الفئات (Class Weights) لمعالجة عدم توازن البيانات

In [10]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# حساب الأوزان لإعطاء أهمية أكبر للفئات الأقل تكراراً
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# تحويل الأوزان إلى Tensor لاستخدامها في دالة الخسارة
weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("أوزان الفئات (Class Weights):\n", weights_tensor)

أوزان الفئات (Class Weights):
 tensor([0.5539, 0.4858, 2.2330, 1.1999, 1.5035, 5.2683])


### الخطوة 4: دالة حساب المقاييس (الدقة و F1 Score)

In [11]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import XLMRobertaForSequenceClassification

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # حساب الدقة (Accuracy)
    acc = accuracy_score(labels, preds)
    # حساب F1-Score (نستخدم weighted بسبب عدم التوازن)
    f1 = f1_score(labels, preds, average='weighted')

    return {'accuracy': acc, 'f1': f1}

# تحميل نموذج XLM-RoBERTa لمهام التصنيف وتحديد عدد الفئات إلى 6
num_labels = 6
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=num_labels)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### الخطوة 5: إعداد Custom Trainer وتدريب النموذج
قمنا ببرمجة `CustomTrainer` لدمج الأوزان المحسوبة في دالة الخسارة. تم تحديد 5 دورات (`epochs=5`) ومعدل تعلم `2e-5`.

In [12]:
from transformers import Trainer, TrainingArguments
from torch import nn

# إنشاء Trainer مخصص يستخدم الـ Class Weights في دالة الخسارة
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # نقل الأوزان إلى نفس الجهاز (CPU أو GPU)
        weight = weights_tensor.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# إعدادات التدريب
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,                 # 5 دورات كما طلبت
    learning_rate=2e-5,                 # معدل التعلم 2e-5
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",              # تم التعديل من evaluation_strategy إلى eval_strategy
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=500,
    load_best_model_at_end=True,
)

# تهيئة المدرب
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

# بدء التدريب
print("جاري بدء التدريب... الرجاء الانتظار (قد يستغرق ذلك بعض الوقت).")
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


جاري بدء التدريب... الرجاء الانتظار (قد يستغرق ذلك بعض الوقت).


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.096066,0.084598,0.966026,0.966583
2,0.105712,0.077678,0.966191,0.966859
3,0.069617,0.079021,0.967473,0.968066
4,0.060436,0.084666,0.967701,0.968251


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=78768, training_loss=0.09915358417769794, metrics={'train_runtime': 4531.3302, 'train_samples_per_second': 278.114, 'train_steps_per_second': 17.383, 'total_flos': 4.144897844542771e+16, 'train_loss': 0.09915358417769794, 'epoch': 4.0})

### الخطوة 6: اختبار النموذج على نصوص لكل شعور

In [13]:
import torch

# جمل اختبارية تغطي كل المشاعر الموجودة
test_sentences = [
    "I feel so lonely and deeply hurt today.",           # Sadness (0)
    "I am incredibly happy and excited for my new job!", # Joy (1)
    "I care about you so much, you mean the world to me.", # Love (2)
    "I am furious and absolutely mad at what you did!",  # Anger (3)
    "I am terrified and panicking about the results.",   # Fear (4)
    "Wow! I never expected this to happen, what a shock!" # Surprise (5)
]

# نقل النموذج لوضع التقييم
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print("--- اختبار النموذج على المشاعر الستة ---\n")
for text in test_sentences:
    # تحضير النص
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    # التنبؤ
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()

    emotion = label_mapping.get(prediction, "Unknown")
    print(f"النص: '{text}'")
    print(f"الشعور المتوقع: {emotion}\n")

--- اختبار النموذج على المشاعر الستة ---

النص: 'I feel so lonely and deeply hurt today.'
الشعور المتوقع: sadness

النص: 'I am incredibly happy and excited for my new job!'
الشعور المتوقع: joy

النص: 'I care about you so much, you mean the world to me.'
الشعور المتوقع: sadness

النص: 'I am furious and absolutely mad at what you did!'
الشعور المتوقع: anger

النص: 'I am terrified and panicking about the results.'
الشعور المتوقع: fear

النص: 'Wow! I never expected this to happen, what a shock!'
الشعور المتوقع: surprise



### الخطوة 7: اختبار النموذج التفاعلي

تتيح لك هذه الخلية إدخال نص خاص بك والحصول على التنبؤ بالشعور من النموذج المدرب.

In [15]:
import torch

# التأكد من أن النموذج في وضع التقييم وعلى الجهاز الصحيح
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print("أدخل النص الذي تريد اختبار النموذج عليه. اكتب 'exit' للخروج.")

while True:
    user_input = input("أدخل نصك هنا: ")
    if user_input.lower() == 'exit':
        break

    # تحضير النص المدخل
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    # التنبؤ
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()

    emotion = label_mapping.get(prediction, "Unknown")
    print(f"النص المدخل: '{user_input}'")
    print(f"الشعور المتوقع: {emotion}\n")

print("تم إنهاء الاختبار التفاعلي.")

أدخل النص الذي تريد اختبار النموذج عليه. اكتب 'exit' للخروج.
أدخل نصك هنا: I  feel a little low the last a few days.
النص المدخل: 'I  feel a little low the last a few days.'
الشعور المتوقع: sadness

أدخل نصك هنا: I hate this guy soo much, i hope to never see him again.
النص المدخل: 'I hate this guy soo much, i hope to never see him again.'
الشعور المتوقع: anger



KeyboardInterrupt: Interrupted by user

### الخطوة 7: حفظ النموذج في Google Drive
تحديث إعدادات النموذج بأسماء المشاعر وحفظه في مجلد `robert 2-1`.

In [16]:
import os

# تحديد مسار الحفظ في جوجل درايف
save_directory = '/content/drive/MyDrive/robert 2-5'
os.makedirs(save_directory, exist_ok=True)

# تحديث إعدادات النموذج لتشمل أسماء المشاعر بدلاً من الأرقام (Labels)
model.config.id2label = label_mapping
model.config.label2id = {v: k for k, v in label_mapping.items()}

# حفظ النموذج
print("جاري حفظ النموذج...")
model.save_pretrained(save_directory)

# حفظ محلل النصوص (Tokenizer)
print("جاري حفظ الـ Tokenizer...")
tokenizer.save_pretrained(save_directory)

print(f"\nتم حفظ النموذج بنجاح في المسار: {save_directory}")
print("الآن يمكنك تحميله واستخدامه مباشرة في المستقبل مع أسماء المشاعر!")

جاري حفظ النموذج...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

جاري حفظ الـ Tokenizer...

تم حفظ النموذج بنجاح في المسار: /content/drive/MyDrive/robert 2-5
الآن يمكنك تحميله واستخدامه مباشرة في المستقبل مع أسماء المشاعر!


### الخطوة 8: استكشاف بيانات Mental Status من Kaggle

In [ ]:
# # تنزيل البيانات من Kaggle وفك ضغطها في مجلد جديد
# !kaggle datasets download -d footsurebead/mental-status --unzip -p mental_status_data

# print("تم تحميل البيانات وفك ضغطها بنجاح!")

In [ ]:
# import pandas as pd
# import os

# # البحث عن ملف CSV داخل المجلد المستخرج
# folder_path = 'mental_status_data'
# csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# if csv_files:
#     file_path = os.path.join(folder_path, csv_files[0])
#     print(f"جاري قراءة الملف: {file_path}\n")

#     # قراءة البيانات
#     df_mental = pd.read_csv(file_path)

#     # 1. طباعة عدد الصفوف
#     print(f"عدد الصفوف الإجمالي في البيانات: {df_mental.shape[0]}")

#     # البحث عن عمود المشاعر/الحالة (غالباً يكون اسمه status أو label أو emotion)
#     target_cols = ['status', 'emotion', 'label', 'mental_state', 'class']
#     emotion_col = None

#     for col in target_cols:
#         if col in [c.lower() for c in df_mental.columns]:
#             emotion_col = [c for c in df_mental.columns if c.lower() == col][0]
#             break

#     if not emotion_col:
#         # إذا لم يتم العثور على اسم مألوف، نستخدم العمود الأخير كافتراض
#         emotion_col = df_mental.columns[-1]
#         print(f"\nلم أتمكن من التعرف على عمود المشاعر تلقائياً، سأستخدم العمود الأخير: '{emotion_col}'")

#     # 2. طباعة عدد الحالات لكل شعور
#     print(f"\nعدد الحالات لكل شعور/حالة (في عمود '{emotion_col}'):")
#     display(df_mental[emotion_col].value_counts())

#     print("\nأول 5 صفوف للتحقق:")
#     display(df_mental.head())
# else:
#     print("لم يتم العثور على ملفات CSV في المجلد.")

In [ ]:
# # الفئات المراد حذفها
# classes_to_remove = ['Anxiety', 'Bipolar', 'Stress', 'Personality disorder', 'Normal']

# # تحويل الفئات إلى أحرف صغيرة لضمان المطابقة
# classes_to_remove_lower = [c.lower() for c in classes_to_remove]

# # فلترة البيانات لإبقاء الصفوف التي لا تنتمي للفئات المحددة
# df_mental = df_mental[~df_mental[emotion_col].str.lower().isin(classes_to_remove_lower)]

# print(f"أبعاد البيانات بعد الحذف: {df_mental.shape}")
# print(f"\nتوزيع الحالات بعد حذف الفئات المحددة:")
# display(df_mental[emotion_col].value_counts())

In [ ]:
# # عرض 10 أمثلة لكل حالة نفسية متبقية
# for status in df_mental[emotion_col].unique():
#     print(f"--- 10 أمثلة لفئة: {status} ---")
#     # نستخدم head(10) لعرض أول 10 صفوف من كل فئة
#     display(df_mental[df_mental[emotion_col] == status][['statement', emotion_col]].head(10))
#     print("\n")

### الخطوة 9: دمج بيانات المشاعر مع بيانات الصحة النفسية لتدريب نموذج شامل لـ 8 فئات

In [ ]:
# import pandas as pd

# # تجهيز بيانات المشاعر لتطابق نفس بنية الأعمدة
# df_emo = df_twitter_emotion[['clean_text', 'emotion_name']].copy()
# df_emo = df_emo.rename(columns={'clean_text': 'text', 'emotion_name': 'label_name'})

# # تجهيز بيانات الصحة النفسية
# df_ment = df_mental[['statement', emotion_col]].copy()
# df_ment = df_ment.rename(columns={'statement': 'text', emotion_col: 'label_name'})

# # توحيد حالة الأحرف (أحرف صغيرة)
# df_ment['label_name'] = df_ment['label_name'].str.lower()
# df_emo['label_name'] = df_emo['label_name'].str.lower()

# # دمج المجموعتين
# df_combined = pd.concat([df_emo, df_ment], ignore_index=True)

# # إنشاء قاموس التسميات الجديد (8 فئات)
# unique_labels = df_combined['label_name'].unique().tolist()
# new_label2id = {label: i for i, label in enumerate(unique_labels)}
# new_id2label = {i: label for label, i in new_label2id.items()}

# # تحويل الأسماء إلى أرقام
# df_combined['label'] = df_combined['label_name'].map(new_label2id)

# print(f"إجمالي عدد العينات بعد الدمج: {len(df_combined)}")
# print("\nخريطة الفئات الجديدة:")
# for k, v in new_id2label.items():
#     print(f"{k}: {v}")

# print("\nتوزيع البيانات على الـ 8 فئات:")
# display(df_combined['label_name'].value_counts())


الخطوة التالية ستكون تقسيم هذه البيانات الجديدة (تدريب واختبار)، وتحويلها (Tokenization)، ثم تدريب نموذج `xlm-roberta-base` بـ 8 فئات (`num_labels=8`). يمكنك تشغيل الخلية أعلاه وسأقوم بكتابة كود التدريب لك في الخطوة القادمة!

### الخطوة 10: تقسيم البيانات الجديدة وتحويلها إلى Dataset

In [ ]:
# from sklearn.model_selection import train_test_split
# from datasets import Dataset

# # تقسيم البيانات
# train_texts_8, val_texts_8, train_labels_8, val_labels_8 = train_test_split(
#     df_combined['text'].tolist(),
#     df_combined['label'].tolist(),
#     test_size=0.2,
#     random_state=42,
#     stratify=df_combined['label'].tolist()
# )

# # تحويل إلى Hugging Face Dataset
# train_dataset_8 = Dataset.from_dict({'text': train_texts_8, 'label': train_labels_8})
# val_dataset_8 = Dataset.from_dict({'text': val_texts_8, 'label': val_labels_8})

# print(f"عدد عينات التدريب: {len(train_dataset_8)}")
# print(f"عدد عينات التقييم: {len(val_dataset_8)}")

### الخطوة 11: Tokenization للبيانات الجديدة

In [ ]:
# from transformers import XLMRobertaTokenizer

# tokenizer_8 = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# def tokenize_function_8(examples):
#     return tokenizer_8(examples['text'], padding='max_length', truncation=True, max_length=128)

# print("جاري تقطيع نصوص التدريب...")
# tokenized_train_8 = train_dataset_8.map(tokenize_function_8, batched=True)
# print("جاري تقطيع نصوص التقييم...")
# tokenized_val_8 = val_dataset_8.map(tokenize_function_8, batched=True)

# tokenized_train_8.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
# tokenized_val_8.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

### الخطوة 12: حساب أوزان الفئات وإعداد النموذج للتدريب

In [ ]:
# import torch
# from transformers import XLMRobertaForSequenceClassification, Trainer, TrainingArguments
# from sklearn.metrics import accuracy_score, f1_score

# # دالة المقاييس
# def compute_metrics_8(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)
#     acc = accuracy_score(labels, preds)
#     f1 = f1_score(labels, preds, average='weighted')
#     return {'accuracy': acc, 'f1': f1}

# # تحميل النموذج بـ 8 فئات (سيتم تصفير الأوزان والبدء من جديد للتخلص من التحيز)
# model_8 = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=8)
# model_8.config.id2label = new_id2label
# model_8.config.label2id = new_label2id

# # إعدادات التدريب
# training_args_8 = TrainingArguments(
#     output_dir='./results_8_classes',
#     num_train_epochs=2,         # دورتين على الأقل أفضل من دورة واحدة
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir='./logs_8',
#     logging_steps=500,
#     load_best_model_at_end=True,
#     fp16=True,                  # الدقة المختلطة لتسريع التدريب
#     dataloader_num_workers=2,   # تسريع جلب البيانات
# )

# # استخدام المدرب الافتراضي (بدون Class Weights)
# trainer_8 = Trainer(
#     model=model_8,
#     args=training_args_8,
#     train_dataset=tokenized_train_8,
#     eval_dataset=tokenized_val_8,
#     compute_metrics=compute_metrics_8
# )

# print("النموذج جاهز للتدريب! قم بتشغيل خلية 'trainer_8.train()' التالية للبدء.")

In [ ]:
# # بدء التدريب (تذكر أن هذا قد يستغرق وقتاً طويلاً)
# trainer_8.train()

### الخطوة 13: اختبار النموذج الشامل (8 فئات)

In [ ]:
# import torch

# # جمل اختبارية جديدة تغطي الـ 8 فئات
# test_sentences_8 = [
#     "I miss my grandfather so much, it breaks my heart.",                             # Sadness (0)
#     "I just won the lottery! This is the best day of my life!",                       # Joy (1)
#     "My partner makes me feel so cherished and adored.",                              # Love (2)
#     "I can't believe they lied to my face, I am absolutely furious!",                 # Anger (3)
#     "I am so scared of walking alone in the dark alley.",                             # Fear (4)
#     "Oh my god, I didn't expect a surprise party at all!",                            # Surprise (5)
#     "Every day feels like a heavy burden, I have no energy to do anything.",          # Depression (6)
#     "I don't want to be here anymore, the pain is unbearable and I want to sleep forever."  # Suicidal (7)
# ]

# # نقل النموذج لوضع التقييم
# model_8.eval()
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_8.to(device)

# print("--- اختبار النموذج على الفئات الثمانية بجمل جديدة ---\n")
# for text in test_sentences_8:
#     # تحضير النص
#     inputs = tokenizer_8(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

#     # التنبؤ
#     with torch.no_grad():
#         outputs = model_8(**inputs)
#         prediction = torch.argmax(outputs.logits, dim=-1).item()

#     emotion = new_id2label.get(prediction, "Unknown")
#     print(f"النص: '{text}'")
#     print(f"التصنيف المتوقع: {emotion}\n")

### حفظ البيانات المنظمة (8 فئات) في Google Drive

In [ ]:
# # حفظ البيانات المدمجة في ملف CSV
# save_path = '/content/drive/MyDrive/combined_emotion_mental_data_RObert2_2.csv'
# df_combined.to_csv(save_path, index=False, encoding='utf-8')

# print(f"تم حفظ البيانات المنظمة بنجاح في: {save_path}")
# print(f"عدد الصفوف المحفوظة: {len(df_combined)}")